In [ ]:
%%capture
!pip install bertopic
import re
import pandas as pd
from datetime import datetime
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
from bertopic import BERTopic
!pip install googletrans==3.1.0a0

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
from googletrans import Translator
translator = Translator(service_urls=[
    'translate.googleapis.com',
  'translate.google.com',
  'translate.google.co.kr',
  'translate.google.co.in',
  'translate.google.co.uk',
])

# language tranlator utility
def trans_msg(msg):
#     if type(msg) != str:
#         return
    trans_m = translator.translate(str(msg), dest='en')
    return str(trans_m.text)

In [ ]:
%%capture
# !pip install -U pip setuptools wheel
# !pip install -U spacy
!python -m spacy download ru_core_news_sm
!python -m spacy download zh_core_web_sm
!python -m spacy download fr_core_news_sm
!python -m spacy download de_core_news_sm
!python -m spacy download it_core_news_sm
!python -m spacy download xx_ent_wiki_sm
!python -m spacy download pl_core_news_sm
!python -m spacy download pt_core_news_sm
!python -m spacy download ro_core_news_sm
!python -m spacy download es_core_news_sm
!python -m spacy download uk_core_news_sm
!python -m spacy download sl_core_news_sm
!python -m spacy download sv_core_news_sm

import spacy
import gensim
from gensim.parsing.preprocessing import remove_stopwords, STOPWORDS

stop_words = stopwords.words('russian')
stop_words.extend(['nan','что', 'это', 'так', 'вот', 'быть', 'как', 'в', '—', 'к', 'на'])
for i in stopwords.fileids():
    stop_words.extend(stopwords.words(str(i)))

en = spacy.load('en_core_web_sm')
sw_spacy = en.Defaults.stop_words
stop_words.extend(list(sw_spacy))

ru = spacy.load('ru_core_news_sm')
sw_spacy_ru = ru.Defaults.stop_words
stop_words.extend(list(sw_spacy_ru))

zh = spacy.load('zh_core_web_sm')
sw_spacy_zh = zh.Defaults.stop_words
stop_words.extend(list(sw_spacy_zh))

fr = spacy.load('fr_core_news_sm')
sw_spacy_fr = fr.Defaults.stop_words
stop_words.extend(list(sw_spacy_fr))

de = spacy.load('de_core_news_sm')
sw_spacy_de = de.Defaults.stop_words
stop_words.extend(list(sw_spacy_de))

it = spacy.load('it_core_news_sm')
sw_spacy_it = it.Defaults.stop_words
stop_words.extend(list(sw_spacy_it))

xx = spacy.load('xx_ent_wiki_sm')
sw_spacy_xx = xx.Defaults.stop_words
stop_words.extend(list(sw_spacy_xx))

pl = spacy.load('pl_core_news_sm')
sw_spacy_pl = pl.Defaults.stop_words
stop_words.extend(list(sw_spacy_pl))

pt = spacy.load('pt_core_news_sm')
sw_spacy_pt = pt.Defaults.stop_words
stop_words.extend(list(sw_spacy_pt))

ro = spacy.load('ro_core_news_sm')
sw_spacy_ro = ro.Defaults.stop_words
stop_words.extend(list(sw_spacy_ro))

es = spacy.load('es_core_news_sm')
sw_spacy_es = es.Defaults.stop_words
stop_words.extend(list(sw_spacy_es))

uk = spacy.load('uk_core_news_sm')
sw_spacy_uk = uk.Defaults.stop_words
stop_words.extend(list(sw_spacy_uk))

sl = spacy.load('sl_core_news_sm')
sw_spacy_sl = sl.Defaults.stop_words
stop_words.extend(list(sw_spacy_sl))

sv = spacy.load('sv_core_news_sm')
sw_spacy_sv = sv.Defaults.stop_words
stop_words.extend(list(sw_spacy_sv))

stop_words.extend(list(STOPWORDS))
stop_words.extend(['suscribirse', 'SuscrÃbete','telegra','subscribe','good','bad','better'])

In [ ]:
mos = pd.read_csv('/kaggle/input/cluster-msgs-data/neuesausrusslanddf.csv')
mos["just_date"] = pd.to_datetime(mos.date).dt.date

mos['msg_without_stopwords'] = mos['cleaned_message'].apply(lambda x: ' '.join([word for word in str(x).split() if word.lower() not in (stop_words)]))
# mos

timestamps = mos.date.to_list()
texts1 = mos.msg_without_stopwords.to_list()

In [ ]:
from bertopic import BERTopic

topic_model = BERTopic(verbose=True, embedding_model="paraphrase-MiniLM-L12-v2", min_topic_size=30)
topics, _ = topic_model.fit_transform(texts1); len(topic_model.get_topic_info())

In [ ]:
freq = topic_model.get_topic_info()
freq['translated_Name'] = freq.Name.apply(trans_msg)
freq['translated_Representation'] = freq.Representation.apply(trans_msg)
freq['translated_Representative_Docs'] = freq.Representative_Docs.apply(trans_msg)

freq.head(10)

In [ ]:
topics_over_time = topic_model.topics_over_time(docs=texts1,
                                                timestamps=timestamps,
                                                global_tuning=True,
                                                evolution_tuning=True,
                                                nr_bins=20)
topic_model.visualize_topics_over_time(topics_over_time, top_n_topics=(len(freq)+1))

In [ ]:
freq.to_csv('neuesausrusslanddf_topics_BERTopic.csv', index=False)